# 04b. Head-Weight Estimation with Observed-Order Plackett-Luce

This notebook estimates the four head weights by matching the **observed within-session order** of validation videos.

It is intentionally different from the pairwise 04b notebook:

- This notebook uses only observed validation rows from `validation_observed_order_head_predictions.parquet`.
- It does not sample unrecommended videos.
- It fits a Plackett-Luce ranking loss within each observed validation session.
- The objective asks whether a linear score of predicted heads can rationalize the platform's exposed order conditional on exposure.


## Identification

Because the true production candidate set is unobserved, this PL estimator does **not** estimate exposure choice against all possible videos. It estimates score weights that best rationalize the observed order among videos that were actually exposed in the validation sessions.

For a validation session with observed videos ordered as `rank_order = 1, ..., K_s`, the score is

$$
    s_{ij} = w_c p_{ij}^{complete} + w_l p_{ij}^{long} + w_r p_{ij}^{rewatch} + w_n p_{ij}^{neg}.
$$

The session likelihood is

$$
    \prod_{r=1}^{K_s-1}
    \frac{\exp(s_{ir}/\tau)}{\sum_{m=r}^{K_s}\exp(s_{im}/\tau)}.
$$

So earlier exposed videos are treated as chosen ahead of later exposed videos, but only within the observed list.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path('/Users/haozhangao/Desktop/RecSys Research')
NEW_CODE = PROJECT_ROOT / 'python_code_new'
OUTPUT_DIR = NEW_CODE / 'outputs'
DOCS_DIR = NEW_CODE / 'docs'
SRC_DIR = NEW_CODE / 'src'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from plackett_luce_weights import (
    HEAD_COLS,
    WEIGHT_NAMES,
    compute_scores,
    fit_pl_weights,
    random_pl_baseline,
    score_pl_metrics,
    pairwise_order_metrics,
    average_session_spearman,
)

OBSERVED_PREDICTIONS_PATH = OUTPUT_DIR / 'validation_observed_order_head_predictions.parquet'
WEIGHTS_PATH = OUTPUT_DIR / 'head_weight_pl_weights.json'
METRICS_PATH = OUTPUT_DIR / 'head_weight_pl_metrics.json'
SCORES_PATH = OUTPUT_DIR / 'pl_session_scores.parquet'
MEAN_HEADS_BY_RANK_PATH = OUTPUT_DIR / 'pl_mean_heads_by_rank.csv'
REPORT_PATH = DOCS_DIR / 'head_weight_pl_estimation_report.md'

TAU = 1.0
LAMBDA_L2 = 1e-3
SESSION_WEIGHTING = 'session'  # choices: 'session', 'item'
PRIMARY_FIT = 'unconstrained'
COMPUTE_EXACT_PAIRWISE_METRICS = True
MAX_RANK_FOR_SUMMARY = 50
WRITE_OUTPUTS = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print('observed predictions path:', OBSERVED_PREDICTIONS_PATH)
print('weights path:', WEIGHTS_PATH)
print('metrics path:', METRICS_PATH)
print('scores path:', SCORES_PATH)
print('report path:', REPORT_PATH)


## 1. Load Observed Validation Predictions

The input comes from `04a_validation_observed_order_head_prediction.ipynb`. It contains one row per observed validation interaction, with predicted engagement heads and observed within-session rank.

Rows with `K_s < 2` cannot contribute to a ranking likelihood, so they are dropped for PL fitting.


In [ ]:
observed = pd.read_parquet(OBSERVED_PREDICTIONS_PATH).copy()

required_cols = [
    'session_key', 'user_id', 'session', 'video_id', 'split',
    'rank_order', 'K_s', 'contributes_to_pl',
] + HEAD_COLS
missing = sorted(set(required_cols) - set(observed.columns))
if missing:
    raise KeyError(f'Observed-order prediction file is missing required columns: {missing}')

if not observed['split'].eq('val').all():
    bad = observed['split'].value_counts(dropna=False).to_dict()
    raise ValueError(f'PL input must contain only validation rows; got split counts: {bad}')

if observed[HEAD_COLS].isna().any().any():
    raise ValueError('Missing predicted head values in observed-order input')
if not observed[HEAD_COLS].apply(lambda s: s.between(0, 1).all()).all():
    raise ValueError('Predicted heads must lie in [0, 1]')

pl_df = observed.loc[observed['contributes_to_pl'].astype(bool)].copy()
pl_df = pl_df.loc[pl_df['K_s'].ge(2)].copy()
pl_df = pl_df.sort_values(['session_key', 'rank_order', 'video_id'], kind='mergesort').reset_index(drop=True)

sample_summary = {
    'rows_total': int(len(observed)),
    'sessions_total': int(observed['session_key'].nunique()),
    'rows_used_for_pl': int(len(pl_df)),
    'sessions_used_for_pl': int(pl_df['session_key'].nunique()),
    'users_used_for_pl': int(pl_df['user_id'].nunique()),
    'sessions_dropped_K_lt_2': int(observed.loc[~observed['contributes_to_pl'].astype(bool), 'session_key'].nunique()),
}

print(json.dumps(sample_summary, indent=2))
display(pl_df[['session_key', 'user_id', 'session', 'video_id', 'rank_order', 'K_s'] + HEAD_COLS].head(20))


## 2. Fit Plackett-Luce Weights

Two fits are estimated:

- **Unconstrained**: all four weights are free. This is the primary saved fit because it directly minimizes the PL likelihood.
- **Sign-constrained diagnostic**: completion, long-view, and rewatch weights are constrained nonnegative; negative-view weight is constrained nonpositive.


In [ ]:
constrained_fit = fit_pl_weights(
    pl_df,
    sign_constraints=True,
    parameterization='bounded',
    tau=TAU,
    lambda_l2=LAMBDA_L2,
    session_weighting=SESSION_WEIGHTING,
    verbose=True,
)

unconstrained_fit = fit_pl_weights(
    pl_df,
    sign_constraints=False,
    tau=TAU,
    lambda_l2=LAMBDA_L2,
    session_weighting=SESSION_WEIGHTING,
    verbose=True,
)

primary_fit = unconstrained_fit if PRIMARY_FIT == 'unconstrained' else constrained_fit
primary_weights = primary_fit.weights
constrained_weights = constrained_fit.weights
unconstrained_weights = unconstrained_fit.weights

def fit_diagnostics(fit):
    return {
        'loss': float(fit.loss),
        'initial_loss': float(fit.initial_loss),
        'converged': bool(fit.converged),
        'message': str(fit.message),
        'num_iterations': int(fit.num_iterations),
        'gradient_norm': float(fit.gradient_norm),
        'parameterization': fit.parameterization,
        'sign_constraints': bool(fit.sign_constraints),
        'weight_norm_l1': float(np.abs(fit.weights).sum()),
        'weight_norm_l2': float(np.linalg.norm(fit.weights)),
        'weights': {name: float(value) for name, value in zip(WEIGHT_NAMES, fit.weights)},
    }

optimization_diagnostics = {
    'constrained': fit_diagnostics(constrained_fit),
    'unconstrained': fit_diagnostics(unconstrained_fit),
}

display(pd.DataFrame([
    {'fit': 'constrained', **optimization_diagnostics['constrained']['weights'], 'loss': constrained_fit.loss, 'converged': constrained_fit.converged},
    {'fit': 'unconstrained', **optimization_diagnostics['unconstrained']['weights'], 'loss': unconstrained_fit.loss, 'converged': unconstrained_fit.converged},
]))


## 3. Score Observed Lists and Compare Baselines

The comparison metrics are all computed on the observed validation lists only.


In [ ]:
score_df = pl_df.copy()
score_df['score_pl_constrained'] = compute_scores(score_df, constrained_weights)
score_df['score_pl_unconstrained'] = compute_scores(score_df, unconstrained_weights)
score_df['score_equal_signed'] = score_df['p_complete'] + score_df['p_long'] + score_df['p_rewatch'] - score_df['p_neg']
score_df['score_single_complete'] = score_df['p_complete']
score_df['score_single_long'] = score_df['p_long']
score_df['score_single_rewatch'] = score_df['p_rewatch']
score_df['score_single_neg_good'] = -score_df['p_neg']

score_cols = [
    'score_pl_unconstrained',
    'score_pl_constrained',
    'score_equal_signed',
    'score_single_complete',
    'score_single_long',
    'score_single_rewatch',
    'score_single_neg_good',
]

pl_metrics = []
for col in score_cols:
    row = {'model': col}
    row.update(score_pl_metrics(score_df, col, tau=TAU))
    pl_metrics.append(row)
random_row = {'model': 'random_ranking'}
random_row.update(random_pl_baseline(score_df))
pl_metrics.append(random_row)
pl_metrics = sorted(pl_metrics, key=lambda x: x['pl_nll'])

display(pd.DataFrame(pl_metrics))


In [ ]:
if COMPUTE_EXACT_PAIRWISE_METRICS:
    pairwise_metrics = []
    for col in score_cols:
        row = {'model': col}
        row.update(pairwise_order_metrics(score_df, score_col=col, tau=TAU))
        pairwise_metrics.append(row)
    pairwise_metrics = sorted(pairwise_metrics, key=lambda x: x['pairwise_order_auc'], reverse=True)
else:
    pairwise_metrics = []

spearman_metrics = []
for col in score_cols:
    row = {'model': col}
    row.update(average_session_spearman(score_df, col))
    spearman_metrics.append(row)
spearman_metrics = sorted(spearman_metrics, key=lambda x: x['mean_spearman_score_vs_early_position'], reverse=True)

if pairwise_metrics:
    display(pd.DataFrame(pairwise_metrics))
display(pd.DataFrame(spearman_metrics))


## 4. Rank Diagnostics

This checks how predicted heads and fitted scores vary by observed position in the session.


In [ ]:
mean_heads_by_rank = (
    score_df.loc[score_df['rank_order'].le(MAX_RANK_FOR_SUMMARY)]
    .groupby('rank_order', observed=True)
    .agg(
        n=('video_id', 'size'),
        sessions=('session_key', 'nunique'),
        mean_p_complete=('p_complete', 'mean'),
        mean_p_long=('p_long', 'mean'),
        mean_p_rewatch=('p_rewatch', 'mean'),
        mean_p_neg=('p_neg', 'mean'),
        mean_score_pl_constrained=('score_pl_constrained', 'mean'),
        mean_score_pl_unconstrained=('score_pl_unconstrained', 'mean'),
        mean_score_equal_signed=('score_equal_signed', 'mean'),
    )
    .reset_index()
)

display(mean_heads_by_rank.head(20))


## 5. Save Weights, Metrics, and Report

The saved weights are the primary downstream score weights. By default, that is the unconstrained PL fit, with constrained weights retained as diagnostics.


In [ ]:
primary_weight_payload = {name: float(value) for name, value in zip(WEIGHT_NAMES, primary_weights)}
constrained_weight_payload = {name: float(value) for name, value in zip(WEIGHT_NAMES, constrained_weights)}
unconstrained_weight_payload = {name: float(value) for name, value in zip(WEIGHT_NAMES, unconstrained_weights)}

empirical_conclusion = {
    'earlier_videos_have_higher_equal_signed_score_on_average': bool(
        mean_heads_by_rank['mean_score_equal_signed'].iloc[0] >= mean_heads_by_rank['mean_score_equal_signed'].iloc[min(1, len(mean_heads_by_rank) - 1)]
    ) if len(mean_heads_by_rank) else False,
    'constrained_weights_sensible_by_construction': bool(
        constrained_weights[0] >= -1e-10 and constrained_weights[1] >= -1e-10 and constrained_weights[2] >= -1e-10 and constrained_weights[3] <= 1e-10
    ),
    'unconstrained_weights_sensible': bool(
        unconstrained_weights[0] >= 0 and unconstrained_weights[1] >= 0 and unconstrained_weights[2] >= 0 and unconstrained_weights[3] <= 0
    ),
    'pl_constrained_beats_random_baseline': bool(
        next(x['pl_nll'] for x in pl_metrics if x['model'] == 'score_pl_constrained') < next(x['pl_nll'] for x in pl_metrics if x['model'] == 'random_ranking')
    ),
}

weights_payload = {
    'weights': primary_weight_payload,
    'head_order': HEAD_COLS,
    'weight_order': WEIGHT_NAMES,
    'score_formula': 'w_complete*p_complete + w_long*p_long + w_rewatch*p_rewatch + w_neg*p_neg',
    'objective': {
        'name': 'observed-order Plackett-Luce ranking loss',
        'uses_unrecommended_negatives': False,
        'tau': float(TAU),
        'lambda_l2': float(LAMBDA_L2),
        'session_weighting': SESSION_WEIGHTING,
        'sign_constraints': None if PRIMARY_FIT == 'unconstrained' else True,
        'parameterization': PRIMARY_FIT,
        'primary_fit': PRIMARY_FIT,
        'constrained_fit_saved_as_diagnostic': True,
    },
    'sample_summary': sample_summary,
    'optimization_diagnostics': optimization_diagnostics,
    'empirical_conclusion': empirical_conclusion,
    'constrained_diagnostic_weights': constrained_weight_payload,
    'unconstrained_diagnostic_weights': unconstrained_weight_payload,
    'note': 'Primary downstream weights are observed-order PL weights. No sampled unrecommended videos are used.',
}

metrics_payload = {
    'sample_summary': sample_summary,
    'pl_metrics': pl_metrics,
    'pairwise_order_metrics': pairwise_metrics,
    'spearman_metrics': spearman_metrics,
    'mean_heads_by_rank_head': mean_heads_by_rank.head(MAX_RANK_FOR_SUMMARY).to_dict(orient='records'),
    'optimization_diagnostics': optimization_diagnostics,
    'empirical_conclusion': empirical_conclusion,
}

report = f"""# Head-Weight PL Estimation Report

## Identification Assumption

Because the true candidate set is unobserved, PL does not estimate exposure against unrecommended videos. It estimates the score weights that best rationalize the observed order conditional on exposure.

## Data

- Observed validation rows used: {sample_summary['rows_used_for_pl']:,}
- Validation sessions used: {sample_summary['sessions_used_for_pl']:,}
- Users used: {sample_summary['users_used_for_pl']:,}
- Sessions dropped because K_s < 2: {sample_summary['sessions_dropped_K_lt_2']:,}

No test sessions and no unrecommended sampled videos are used.

## Objective

Plackett-Luce observed-order loss with {SESSION_WEIGHTING} weighting.

- tau: {TAU}
- lambda_l2: {LAMBDA_L2}
- primary fit: {PRIMARY_FIT}

## Primary Weights

```json
{json.dumps(primary_weight_payload, indent=2)}
```

## Constrained Diagnostic Weights

```json
{json.dumps(constrained_weight_payload, indent=2)}
```

## Empirical Conclusion

```json
{json.dumps(empirical_conclusion, indent=2)}
```

## Main Diagnostics

PL metrics, pairwise order metrics, Spearman correlations, and mean heads by rank are saved in `{METRICS_PATH.relative_to(PROJECT_ROOT)}` and `{MEAN_HEADS_BY_RANK_PATH.relative_to(PROJECT_ROOT)}`.
"""

if WRITE_OUTPUTS:
    WEIGHTS_PATH.write_text(json.dumps(weights_payload, indent=2) + '\n')
    METRICS_PATH.write_text(json.dumps(metrics_payload, indent=2, default=float) + '\n')
    score_df.to_parquet(SCORES_PATH, index=False)
    mean_heads_by_rank.to_csv(MEAN_HEADS_BY_RANK_PATH, index=False)
    REPORT_PATH.write_text(report)
    print('saved weights:', WEIGHTS_PATH)
    print('saved metrics:', METRICS_PATH)
    print('saved scores:', SCORES_PATH)
    print('saved mean heads by rank:', MEAN_HEADS_BY_RANK_PATH)
    print('saved report:', REPORT_PATH)
else:
    print('WRITE_OUTPUTS is False; skipped saving.')

display(pd.DataFrame([primary_weight_payload], index=[PRIMARY_FIT]))
